# Data/API Management Colab

Notebook operativo comune per `Database Finanziario`, cartella privata `API`, registry provider, stato credenziali mascherato e contratti consumabili da app/notebook.

Policy: Drive-first, cache-second, API-last. Nessun valore segreto viene scritto negli output.


In [ ]:
# Parameters
FINANCIAL_DB_ROOT_OVERRIDE = ""
API_ROOT_OVERRIDE = ""
MAX_INVENTORY_FILES = 5000
RUN_INCREMENTAL_REFRESH = False
REFRESH_MAX_SYMBOLS = 25
REFRESH_STALE_HOURS = 24 * 7
SAMPLE_PRICE_TICKER = "ENEL.MI"


In [ ]:
# ============================================================
# Portable Research Platform Bootstrap - Data/API v1.0
# ============================================================
from pathlib import Path
import os
import sys
import subprocess

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    ENVIRONMENT = "colab"
except Exception:
    ENVIRONMENT = "local"

cwd = Path.cwd().resolve()
project_candidates = [
    Path(os.environ.get("RESEARCH_PLATFORM_ROOT", "")).expanduser() if os.environ.get("RESEARCH_PLATFORM_ROOT") else None,
    Path("/content/drive/MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive"),
    Path("/content/drive/MyDrive/machine-learning-for-trading/research_platform_definitive"),
    Path("/content/machine-learning-for-trading/research_platform_definitive"),
    Path("/content/research_platform_definitive"),
    cwd,
    *cwd.parents,
]

PROJECT_ROOT = None
for candidate in [c for c in project_candidates if c is not None]:
    candidate = candidate.expanduser().resolve()
    direct = candidate / "src" / "research_platform_core"
    nested = candidate / "research_platform_definitive" / "src" / "research_platform_core"
    if direct.exists():
        PROJECT_ROOT = candidate
        break
    if nested.exists():
        PROJECT_ROOT = candidate / "research_platform_definitive"
        break

if PROJECT_ROOT is None and os.environ.get("RESEARCH_PLATFORM_AUTO_CLONE") == "1":
    git_url = os.environ.get("RESEARCH_PLATFORM_GIT_URL", "https://github.com/TheGenesisAIStory/ml-trading-thesis-bot.git")
    target = Path("/content/machine-learning-for-trading")
    if not target.exists():
        subprocess.run(["git", "clone", git_url, str(target)], check=False)
    if (target / "research_platform_definitive" / "src" / "research_platform_core").exists():
        PROJECT_ROOT = target / "research_platform_definitive"

if PROJECT_ROOT is None:
    searched = "\n".join(f"- {p}" for p in project_candidates if p is not None)
    raise FileNotFoundError(
        "research_platform_definitive not found. Sync the repo to Drive or set RESEARCH_PLATFORM_ROOT.\n"
        f"Searched:\n{searched}"
    )

for rel in ["", "src", "company_valuation/src", "portfolio_analysis/src"]:
    path = str(PROJECT_ROOT / rel)
    if path not in sys.path:
        sys.path.insert(0, path)

financial_candidates = [
    Path(FINANCIAL_DB_ROOT_OVERRIDE).expanduser() if FINANCIAL_DB_ROOT_OVERRIDE else None,
    Path(os.environ.get("FINANCIAL_DB_ROOT", "")).expanduser() if os.environ.get("FINANCIAL_DB_ROOT") else None,
    Path("/content/drive/MyDrive/Database Finanziario"),
    Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario",
]
FINANCIAL_DB_ROOT = next((p for p in financial_candidates if p is not None and p.exists()), financial_candidates[0] or Path("/content/drive/MyDrive/Database Finanziario"))
API_ROOT = Path(API_ROOT_OVERRIDE).expanduser() if API_ROOT_OVERRIDE else Path(os.environ.get("API_CREDENTIALS_ROOT", FINANCIAL_DB_ROOT / "API")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("RESEARCH_PLATFORM_OUTPUT_ROOT", PROJECT_ROOT / "output")).expanduser()
CACHE_ROOT = Path(os.environ.get("RESEARCH_PLATFORM_LOCAL_CACHE", OUTPUT_ROOT / "data_cache")).expanduser()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "RESEARCH_PLATFORM_ROOT": str(PROJECT_ROOT),
    "FINANCIAL_DB_ROOT": str(FINANCIAL_DB_ROOT),
    "DB_BASE": str(FINANCIAL_DB_ROOT),
    "DATA_PATH": str(FINANCIAL_DB_ROOT),
    "API_CREDENTIALS_ROOT": str(API_ROOT),
    "RESEARCH_PLATFORM_OUTPUT_ROOT": str(OUTPUT_ROOT),
    "RESEARCH_PLATFORM_LOCAL_CACHE": str(CACHE_ROOT),
})

print(f"Environment: {ENVIRONMENT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FINANCIAL_DB_ROOT: {FINANCIAL_DB_ROOT} | exists={FINANCIAL_DB_ROOT.exists()}")
print(f"API_ROOT: {API_ROOT} | exists={API_ROOT.exists()}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")


In [ ]:
# Load shared Data/API status
import pandas as pd
from IPython.display import display

from research_platform_core.data_platform import (
    dataset_status,
    provider_fallback_plan,
    read_dataset_drive_first,
    write_data_platform_status,
)
from research_platform_core.api_management import (
    accepted_api_env_vars,
    api_control_status,
    apply_env_text_to_session,
    build_env_template,
    write_api_control_status,
)

DATA_STATUS = dataset_status(FINANCIAL_DB_ROOT, max_files=MAX_INVENTORY_FILES)
API_STATUS = api_control_status(FINANCIAL_DB_ROOT, API_ROOT)

print("Data summary")
display(DATA_STATUS.get("summary", pd.DataFrame()))
print("API summary")
display(API_STATUS.get("summary", pd.DataFrame()))
print("Credential status (masked)")
display(API_STATUS.get("credential_status", pd.DataFrame()))


In [ ]:
# Session-only API key loader
# Paste .env-style text below only when needed. Values stay in os.environ for this runtime.
API_ENV_TEXT = """# FRED_API_KEY=
# ALPHA_VANTAGE_API_KEY=
"""

providers = API_STATUS.get("api_providers", pd.DataFrame())
allowed_env_vars = accepted_api_env_vars(providers)
if API_ENV_TEXT.strip() and "=" in API_ENV_TEXT:
    loaded = apply_env_text_to_session(API_ENV_TEXT, allowed_env_vars=allowed_env_vars)
    print(f"Loaded {len(loaded)} credential(s) into this notebook session.")
    display(loaded)
else:
    print("No session credentials loaded. Fill API_ENV_TEXT and rerun this cell when an API fallback is required.")

print("Accepted env vars")
print(build_env_template(providers))


In [ ]:
# Provider fallback plan
fallback = provider_fallback_plan(FINANCIAL_DB_ROOT)
display(fallback)


In [ ]:
# Drive-first sample dataset read
prices, meta = read_dataset_drive_first(
    FINANCIAL_DB_ROOT,
    "prices",
    SAMPLE_PRICE_TICKER,
    max_age_hours=REFRESH_STALE_HOURS,
    allow_stale=True,
)
print(meta)
display(prices.tail() if hasattr(prices, "tail") else prices)


In [ ]:
# Optional stale-aware API refresh
# Leave RUN_INCREMENTAL_REFRESH=False unless you explicitly want provider calls.
from research_platform_core.data_platform import refresh_europe_stoxx_prices_incremental

if RUN_INCREMENTAL_REFRESH:
    manifest = refresh_europe_stoxx_prices_incremental(
        FINANCIAL_DB_ROOT,
        max_symbols=int(REFRESH_MAX_SYMBOLS),
        stale_hours=int(REFRESH_STALE_HOURS),
        force=False,
    )
    display(manifest)
else:
    print("Skipped. Set RUN_INCREMENTAL_REFRESH=True in Parameters to refresh stale/missing price files.")


In [ ]:
# Write app/API contracts
# Contracts are safe to commit: they contain tables, paths and masked status, not secret values.
data_paths = write_data_platform_status(FINANCIAL_DB_ROOT, OUTPUT_ROOT, max_files=MAX_INVENTORY_FILES)
api_paths = write_api_control_status(FINANCIAL_DB_ROOT, OUTPUT_ROOT, API_ROOT)

print("Data platform contract paths")
for name, path in data_paths.items():
    print(f"{name}: {path}")
print("
API control contract paths")
for name, path in api_paths.items():
    print(f"{name}: {path}")


## Local app entrypoint

Run from the repository root:

```bash
streamlit run research_platform_definitive/data_api_app.py
```

Or open the same view inside the main app at `research_platform_app/pages/11_Data_API_Control_Center.py`.
